In [0]:

payroll_df = spark.read.table("global_tech_sliver.transformed_data_silver.tf_payroll")

from pyspark.sql.functions import col

fact_payroll = payroll_df.select(
    "employee_id",
    "company_id",
    "department_id",
    "pay_date",
    "gross_salary",
    "bonus",
    "overtime_pay",
    "commission",
    "allowances",
    "net_salary",
    "total_compensation",
)
spark.sql("CREATE SCHEMA IF NOT EXISTS global_tech_gold.global_tech_fact_tables")

fact_payroll.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable("global_tech_gold.global_tech_fact_tables.fact_payroll")

display(fact_payroll)

In [0]:
from pyspark.sql.functions import col

gl_df = spark.read.table("global_tech_sliver.transformed_data_silver.tf_gl")
acc_df = spark.read.table("global_tech_sliver.transformed_data_silver.tf_accounts")

fact_finance = gl_df.join(acc_df, "account_id") \
    .withColumn("amount",col("debit_amount") - col("credit_amount")).select(
        "company_id",
        "department_id",
        "account_sk",
        "posting_date",
        "amount",
        "account_type",
        "category"
    )

spark.sql("CREATE SCHEMA IF NOT EXISTS global_tech_gold.global_tech_fact_tables")

fact_finance.write.format("delta").option("mergeSchema","true").mode("overwrite").saveAsTable("global_tech_gold.global_tech_fact_tables.fact_finance")

display(fact_finance)